# Unified PRMP Benchmark: 5 Variants x 5 Seeds x 3 Tasks

**Predictive Residual Message Passing (PRMP)** for Heterogeneous GNNs over FK-linked tables.

**Hypothesis:** In heterogeneous GNNs over FK-linked tables, predicting child features from parent features and passing *residuals* instead of raw features improves learning, especially for predictable FK links.

**5 Model Variants:**
1. **Standard** HeteroSAGEConv (baseline)
2. **PRMP** (predict-subtract-aggregate)
3. **Wide** (parameter-matched standard — same param count as PRMP)
4. **Auxiliary MLP** (extra capacity, no subtraction — tests if benefit is from extra params)
5. **Random Frozen** (random predictions — tests if any subtraction helps)

**3 Tasks:** `rel-amazon/review-rating` (regression, MAE), `rel-f1/result-position` (regression, MAE), `rel-f1/result-dnf` (classification, Average Precision)

This notebook demonstrates the model architectures on synthetic data, then visualizes pre-computed benchmark results from 75 runs (5 variants x 5 seeds x 3 tasks).

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT pre-installed on Colab
_pip('loguru==0.7.3')

# Core packages: pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1',
         'scipy==1.15.3', 'matplotlib==3.10.0', 'tqdm==4.67.3')
    _pip('torch==2.6.0+cpu', '--extra-index-url', 'https://download.pytorch.org/whl/cpu')

## Imports

In [ ]:
import gc
import json
import math
import os
import sys
import time
import warnings
from collections import defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from loguru import logger
from scipy import stats
from sklearn.linear_model import Ridge
from sklearn.metrics import average_precision_score, mean_absolute_error, r2_score
from tqdm import tqdm

warnings.filterwarnings("ignore")

# Pure PyTorch scatter_mean (replaces torch_scatter dependency)
def scatter_mean(src, index, dim=0, dim_size=None):
    """Compute mean of src elements sharing the same index."""
    if dim_size is None:
        dim_size = int(index.max()) + 1 if len(index) > 0 else 0
    out = torch.zeros(dim_size, src.shape[1], dtype=src.dtype, device=src.device)
    count = torch.zeros(dim_size, 1, dtype=src.dtype, device=src.device)
    idx_expand = index.unsqueeze(1).expand_as(src)
    out.scatter_add_(dim, idx_expand, src)
    count.scatter_add_(dim, index.unsqueeze(1).expand(-1, 1),
                       torch.ones(src.shape[0], 1, dtype=src.dtype, device=src.device))
    return out / count.clamp(min=1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## Load Pre-computed Benchmark Results

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-outputs/ai-invention-b2d5b0-predictive-residual-message-passing-filt/main/experiment_iter6_unified_prmp_be/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

data = load_data()
print(f"Loaded data with {len(data['datasets'])} datasets")
for ds in data['datasets']:
    print(f"  {ds['dataset']}: {len(ds['examples'])} examples")

## Configuration

All tunable hyperparameters. The demo uses smaller values for fast execution; original values are commented.

In [ ]:
# ============================================================
# CONFIGURATION — Demo values (originals commented)
# ============================================================
CONFIG = {
    "hidden_dim": 32,           # Original: 128
    "num_gnn_layers": 2,        # Original: 2
    "lr": 0.001,                # Original: 0.001
    "max_epochs": 20,           # Original: 200
    "patience": 5,              # Original: 20
    "seeds": [42],              # Original: [42, 123, 456, 789, 1024]
    "gradient_clip": 1.0,       # Original: 1.0
    "weight_decay": 1e-5,       # Original: 1e-5
    "prediction_mlp_hidden": 16, # Original: 64
}

# Variants to train in the demo (all 5 shown in results visualization)
DEMO_VARIANTS = ["standard", "prmp", "auxiliary_mlp"]  # Original: all 5
ALL_VARIANTS = ["standard", "prmp", "wide", "auxiliary_mlp", "random_frozen"]

# Synthetic graph size
N_REVIEWS = 200    # Number of child nodes (reviews/results)
N_PRODUCTS = 30    # Number of parent nodes type 1
N_CUSTOMERS = 40   # Number of parent nodes type 2
FEAT_DIM = 10      # Feature dimension per node

print("Config:", json.dumps(CONFIG, indent=2))
print(f"Demo variants: {DEMO_VARIANTS}")
print(f"Synthetic graph: {N_REVIEWS} reviews, {N_PRODUCTS} products, {N_CUSTOMERS} customers")

## Heterogeneous Graph Data Structure

Lightweight container for heterogeneous graphs with node features, edge indices, masks, and FK edge annotations.

In [ ]:
# ============================================================
# HETEROGENEOUS GRAPH DATA STRUCTURE
# ============================================================
class HeteroGraphData:
    """Lightweight heterogeneous graph container."""

    def __init__(self):
        self.node_features: dict[str, torch.Tensor] = {}
        self.edge_index: dict[tuple, torch.Tensor] = {}
        self.target: torch.Tensor | None = None
        self.target_node_type: str = ""
        self.train_mask: torch.Tensor | None = None
        self.val_mask: torch.Tensor | None = None
        self.test_mask: torch.Tensor | None = None
        self.num_nodes: dict[str, int] = {}
        self.fk_edges: list[tuple] = []
        self.target_mean: float = 0.0
        self.target_std: float = 1.0

    def to(self, device: torch.device) -> "HeteroGraphData":
        g = HeteroGraphData()
        g.node_features = {k: v.to(device) for k, v in self.node_features.items()}
        g.edge_index = {k: v.to(device) for k, v in self.edge_index.items()}
        if self.target is not None:
            g.target = self.target.to(device)
        if self.train_mask is not None:
            g.train_mask = self.train_mask.to(device)
        if self.val_mask is not None:
            g.val_mask = self.val_mask.to(device)
        if self.test_mask is not None:
            g.test_mask = self.test_mask.to(device)
        g.num_nodes = dict(self.num_nodes)
        g.target_node_type = self.target_node_type
        g.fk_edges = list(self.fk_edges)
        g.target_mean = self.target_mean
        g.target_std = self.target_std
        return g

    @property
    def node_types(self) -> list[str]:
        return list(self.node_features.keys())

    @property
    def edge_types(self) -> list[tuple]:
        return list(self.edge_index.keys())

    @property
    def in_channels_dict(self) -> dict[str, int]:
        return {k: v.shape[1] for k, v in self.node_features.items()}

print("HeteroGraphData defined")

## Model Architectures

- **TabularEncoder**: Per-node-type MLP mapping raw features to hidden-dim embeddings
- **HeteroSAGEConvLayer**: Standard heterogeneous message passing with scatter aggregation
- **PRMPConvLayer**: PRMP convolution — for FK edges, parent predicts child features via MLP, then residuals (actual - predicted) are aggregated. Supports `no_subtraction` (auxiliary MLP) and `freeze_predictors` (random frozen) modes
- **HeteroGNN**: Full model combining encoder, conv layers, and prediction head

In [ ]:
# ============================================================
# MODEL ARCHITECTURES
# ============================================================
class TabularEncoder(nn.Module):
    """Per-node-type MLP: raw features -> hidden_dim embeddings."""
    def __init__(self, in_channels_dict: dict[str, int], hidden_dim: int):
        super().__init__()
        self.encoders = nn.ModuleDict()
        for ntype, in_dim in in_channels_dict.items():
            self.encoders[ntype] = nn.Sequential(
                nn.Linear(in_dim, hidden_dim), nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
    def forward(self, x_dict):
        return {nt: self.encoders[nt](x) for nt, x in x_dict.items()}


class HeteroSAGEConvLayer(nn.Module):
    """Standard heterogeneous SAGEConv layer with scatter aggregation."""
    def __init__(self, hidden_dim, edge_types, node_types):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.msg_linears = nn.ModuleDict()
        for et in edge_types:
            self.msg_linears["__".join(et)] = nn.Linear(hidden_dim, hidden_dim)
        self.self_linears = nn.ModuleDict()
        self.combine_linears = nn.ModuleDict()
        for nt in node_types:
            self.self_linears[nt] = nn.Linear(hidden_dim, hidden_dim)
            self.combine_linears[nt] = nn.Linear(2 * hidden_dim, hidden_dim)

    def forward(self, x_dict, edge_index_dict):
        agg_dict = {nt: [] for nt in x_dict}
        for et, ei in edge_index_dict.items():
            key = "__".join(et)
            if key not in self.msg_linears:
                continue
            src_type, _, dst_type = et
            msg = self.msg_linears[key](x_dict[src_type][ei[0]])
            agg = scatter_mean(msg, ei[1], dim=0, dim_size=x_dict[dst_type].shape[0])
            agg_dict[dst_type].append(agg)
        out = {}
        for nt, x in x_dict.items():
            self_out = self.self_linears[nt](x)
            if agg_dict[nt]:
                neigh = torch.stack(agg_dict[nt]).mean(dim=0)
                out[nt] = F.relu(self.combine_linears[nt](torch.cat([self_out, neigh], -1)))
            else:
                out[nt] = F.relu(self_out)
        return out


class PRMPConvLayer(nn.Module):
    """PRMP convolution: predict-subtract-aggregate on FK edges + standard on all.

    For FK edge (child -> parent):
      1. Standard message aggregation (same as HeteroSAGEConv)
      2. ADDITIONALLY: parent predicts child features via MLP
      3. Residual = actual_child - predicted_child
      4. Aggregate residuals to parent
      5. Both signals combined in the update
    """
    def __init__(self, hidden_dim, edge_types, node_types, fk_edges,
                 pred_mlp_hidden=64, freeze_predictors=False, no_subtraction=False):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.fk_edge_set = {tuple(e) for e in fk_edges}
        self.no_subtraction = no_subtraction
        self.msg_linears = nn.ModuleDict()
        for et in edge_types:
            self.msg_linears["__".join(et)] = nn.Linear(hidden_dim, hidden_dim)
        self.pred_mlps = nn.ModuleDict()
        for et in edge_types:
            if tuple(et) in self.fk_edge_set:
                mlp = nn.Sequential(
                    nn.Linear(hidden_dim, pred_mlp_hidden), nn.ReLU(),
                    nn.Linear(pred_mlp_hidden, hidden_dim),
                )
                if freeze_predictors:
                    for p in mlp.parameters():
                        p.requires_grad = False
                self.pred_mlps["__".join(et)] = mlp
        self.self_linears = nn.ModuleDict()
        self.combine_linears = nn.ModuleDict()
        for nt in node_types:
            self.self_linears[nt] = nn.Linear(hidden_dim, hidden_dim)
            self.combine_linears[nt] = nn.Linear(2 * hidden_dim, hidden_dim)

    def forward(self, x_dict, edge_index_dict):
        agg_dict = {nt: [] for nt in x_dict}
        for et, ei in edge_index_dict.items():
            src_type, rel, dst_type = et
            key = "__".join(et)
            if key in self.msg_linears:
                msg = self.msg_linears[key](x_dict[src_type][ei[0]])
                agg = scatter_mean(msg, ei[1], dim=0, dim_size=x_dict[dst_type].shape[0])
                agg_dict[dst_type].append(agg)
            if tuple(et) in self.fk_edge_set and key in self.pred_mlps:
                child_x, parent_x = x_dict[src_type], x_dict[dst_type]
                predicted_child = self.pred_mlps[key](parent_x[ei[1]])
                actual_child = child_x[ei[0]]
                if self.no_subtraction:
                    enriched = actual_child + predicted_child
                    prmp_agg = scatter_mean(enriched, ei[1], dim=0, dim_size=parent_x.shape[0])
                else:
                    residuals = actual_child - predicted_child
                    prmp_agg = scatter_mean(residuals, ei[1], dim=0, dim_size=parent_x.shape[0])
                agg_dict[dst_type].append(prmp_agg)
        out = {}
        for nt, x in x_dict.items():
            self_out = self.self_linears[nt](x)
            if agg_dict[nt]:
                neigh = torch.stack(agg_dict[nt]).mean(dim=0)
                out[nt] = F.relu(self.combine_linears[nt](torch.cat([self_out, neigh], -1)))
            else:
                out[nt] = F.relu(self_out)
        return out


class PredictionHead(nn.Module):
    """MLP prediction head: hidden_dim -> 1."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
        )
    def forward(self, x):
        return self.head(x).squeeze(-1)


class HeteroGNN(nn.Module):
    """Complete heterogeneous GNN with configurable conv layers."""
    def __init__(self, in_channels_dict, hidden_dim, num_layers, edge_types,
                 node_types, variant="standard", fk_edges=None, pred_mlp_hidden=64):
        super().__init__()
        self.variant = variant
        self.encoder = TabularEncoder(in_channels_dict, hidden_dim)
        self.convs = nn.ModuleList()
        for _ in range(num_layers):
            if variant in ("standard", "wide"):
                layer = HeteroSAGEConvLayer(hidden_dim, edge_types, node_types)
            elif variant == "prmp":
                layer = PRMPConvLayer(hidden_dim, edge_types, node_types,
                                     fk_edges or [], pred_mlp_hidden)
            elif variant == "auxiliary_mlp":
                layer = PRMPConvLayer(hidden_dim, edge_types, node_types,
                                     fk_edges or [], pred_mlp_hidden, no_subtraction=True)
            elif variant == "random_frozen":
                layer = PRMPConvLayer(hidden_dim, edge_types, node_types,
                                     fk_edges or [], pred_mlp_hidden, freeze_predictors=True)
            else:
                raise ValueError(f"Unknown variant: {variant}")
            self.convs.append(layer)
        self.head = PredictionHead(hidden_dim)

    def encode(self, x_dict, edge_index_dict):
        h = self.encoder(x_dict)
        for conv in self.convs:
            h = conv(h, edge_index_dict)
        return h

    def forward(self, x_dict, edge_index_dict, target_node_type):
        h = self.encode(x_dict, edge_index_dict)
        return self.head(h[target_node_type])


def _count_params(model):
    return sum(p.numel() for p in model.parameters())

print("All model classes defined")

## Generate Synthetic Heterogeneous Graph

Creates a small heterogeneous graph mimicking the Amazon review structure: review nodes (child) connected to product and customer nodes (parents) via FK edges. Target is a synthetic "rating" derived from a linear combination of parent features plus noise, making the FK prediction mechanism relevant.

In [ ]:
# ============================================================
# SYNTHETIC GRAPH GENERATION
# ============================================================
def generate_synthetic_graph(n_reviews, n_products, n_customers, feat_dim, seed=42):
    """Generate a synthetic heterogeneous graph mimicking Amazon reviews."""
    rng = np.random.RandomState(seed)

    # Generate features
    review_feats = torch.tensor(rng.randn(n_reviews, feat_dim), dtype=torch.float32)
    product_feats = torch.tensor(rng.randn(n_products, feat_dim), dtype=torch.float32)
    customer_feats = torch.tensor(rng.randn(n_customers, feat_dim), dtype=torch.float32)

    # Assign each review to a product and customer (FK links)
    product_ids = torch.tensor(rng.randint(0, n_products, n_reviews), dtype=torch.long)
    customer_ids = torch.tensor(rng.randint(0, n_customers, n_reviews), dtype=torch.long)
    review_idx = torch.arange(n_reviews, dtype=torch.long)

    # Target: rating = f(product_features, customer_features) + noise
    # This makes the FK prediction mechanism relevant
    w = torch.tensor(rng.randn(feat_dim), dtype=torch.float32)
    target = (product_feats[product_ids] @ w + customer_feats[customer_ids] @ w * 0.5
              + torch.tensor(rng.randn(n_reviews) * 0.5, dtype=torch.float32))

    graph = HeteroGraphData()
    graph.node_features = {
        "review": review_feats,
        "product": product_feats,
        "customer": customer_feats,
    }
    graph.num_nodes = {"review": n_reviews, "product": n_products, "customer": n_customers}
    graph.edge_index = {
        ("review", "belongs_to", "product"): torch.stack([review_idx, product_ids]),
        ("review", "written_by", "customer"): torch.stack([review_idx, customer_ids]),
        ("product", "has_review", "review"): torch.stack([product_ids, review_idx]),
        ("customer", "wrote", "review"): torch.stack([customer_ids, review_idx]),
    }
    graph.fk_edges = [
        ("review", "belongs_to", "product"),
        ("review", "written_by", "customer"),
    ]
    graph.target_mean = float(target.mean())
    graph.target_std = float(target.std())
    graph.target = (target - graph.target_mean) / max(graph.target_std, 1e-6)
    graph.target_node_type = "review"

    # 70/15/15 split
    n_train = int(n_reviews * 0.7)
    n_val = int(n_reviews * 0.15)
    graph.train_mask = torch.zeros(n_reviews, dtype=torch.bool)
    graph.val_mask = torch.zeros(n_reviews, dtype=torch.bool)
    graph.test_mask = torch.zeros(n_reviews, dtype=torch.bool)
    graph.train_mask[:n_train] = True
    graph.val_mask[n_train:n_train + n_val] = True
    graph.test_mask[n_train + n_val:] = True

    return graph

graph = generate_synthetic_graph(N_REVIEWS, N_PRODUCTS, N_CUSTOMERS, FEAT_DIM)
print(f"Graph nodes: {graph.num_nodes}")
print(f"Graph edge types: {len(graph.edge_types)}")
print(f"FK edges: {graph.fk_edges}")
print(f"Train/Val/Test: {graph.train_mask.sum()}/{graph.val_mask.sum()}/{graph.test_mask.sum()}")

## Training Demo

Train each variant on the synthetic graph and compare test MAE. This demonstrates the full training pipeline: model construction, training with early stopping, and evaluation.

In [ ]:
# ============================================================
# TRAINING & EVALUATION
# ============================================================
def evaluate_model(model, graph, mask, return_preds=False):
    """Evaluate model on masked subset (regression MAE)."""
    model.eval()
    with torch.no_grad():
        pred = model(graph.node_features, graph.edge_index, graph.target_node_type)
        p = pred[mask]
        t = graph.target[mask]
        p_np = (p * graph.target_std + graph.target_mean).cpu().numpy()
        t_np = (t * graph.target_std + graph.target_mean).cpu().numpy()
        mae = float(mean_absolute_error(t_np, p_np))
    if return_preds:
        return mae, p_np, t_np
    return mae


def train_single_run(variant, seed, config, graph):
    """Train one model variant. Returns (result_dict, test_preds, test_targets)."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    g = graph.to(DEVICE)
    edge_types = [list(et) for et in g.edge_types]
    node_types = g.node_types
    fk_edges = [list(e) for e in g.fk_edges]

    model = HeteroGNN(
        g.in_channels_dict, config["hidden_dim"], config["num_gnn_layers"],
        edge_types, node_types, variant, fk_edges, config["prediction_mlp_hidden"],
    ).to(DEVICE)
    num_params = _count_params(model)

    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad],
        lr=config["lr"], weight_decay=config["weight_decay"],
    )
    criterion = nn.MSELoss()

    best_val = float("inf")
    best_epoch = 0
    patience_ctr = 0
    best_state = None
    train_losses = []

    for epoch in range(config["max_epochs"]):
        model.train()
        optimizer.zero_grad()
        pred = model(g.node_features, g.edge_index, g.target_node_type)
        loss = criterion(pred[g.train_mask], g.target[g.train_mask])
        if torch.isnan(loss) or torch.isinf(loss):
            break
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), config["gradient_clip"])
        optimizer.step()
        train_losses.append(float(loss))

        val_mae = evaluate_model(model, g, g.val_mask)
        if val_mae < best_val:
            best_val = val_mae
            best_epoch = epoch
            patience_ctr = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1
            if patience_ctr >= config["patience"]:
                break

    if best_state is not None:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    test_mae, test_preds, test_targets = evaluate_model(model, g, g.test_mask, return_preds=True)

    result = {
        "variant": variant, "seed": seed, "best_epoch": best_epoch,
        "best_val_mae": best_val, "test_mae": test_mae,
        "num_params": num_params, "train_losses": train_losses,
    }
    del model, best_state
    gc.collect()
    return result, test_preds, test_targets


# --- Run training for demo variants ---
print(f"Training {len(DEMO_VARIANTS)} variants x {len(CONFIG['seeds'])} seeds on synthetic graph...\n")
demo_results = []
demo_preds = {}

for variant in DEMO_VARIANTS:
    for seed in CONFIG["seeds"]:
        t0 = time.time()
        res, preds, targets = train_single_run(variant, seed, CONFIG, graph)
        elapsed = time.time() - t0
        demo_results.append(res)
        demo_preds[variant] = (preds, targets)
        print(f"  {variant:15s} seed={seed}: test_mae={res['test_mae']:.4f}, "
              f"best_epoch={res['best_epoch']}, params={res['num_params']}, "
              f"time={elapsed:.1f}s")

print("\nDemo training complete!")

## Training Loss Curves (Demo)

In [ ]:
# ============================================================
# TRAINING LOSS CURVES (DEMO)
# ============================================================
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
for res in demo_results:
    ax.plot(res["train_losses"], label=f"{res['variant']} (MAE={res['test_mae']:.3f})")
ax.set_xlabel("Epoch")
ax.set_ylabel("Training Loss (MSE)")
ax.set_title("Training Loss Curves on Synthetic Graph")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Full Benchmark Results (Pre-computed)

The following visualizations use the pre-computed results from 75 runs (5 variants x 5 seeds x 3 tasks) on real Amazon and F1 datasets.

In [ ]:
# ============================================================
# RESULTS SUMMARY TABLE
# ============================================================
meta = data["metadata"]
stat_analysis = meta["statistical_analysis"]

print("=" * 80)
print("FULL BENCHMARK RESULTS (5 variants x 5 seeds x 3 tasks = 75 runs)")
print(f"Total time: {meta['total_time_seconds']:.1f}s")
print(f"Successful runs: {meta['total_runs_successful']}/{meta['total_runs_attempted']}")
print("=" * 80)

for task_key, analysis in stat_analysis.items():
    metric = analysis["metric"]
    lower = analysis["lower_better"]
    best = analysis["best_variant"]
    direction = "lower is better" if lower else "higher is better"
    print(f"\n{'─' * 60}")
    print(f"Task: {task_key} | Metric: {metric} ({direction})")
    print(f"Best variant: {best}")
    print(f"{'─' * 60}")
    print(f"  {'Variant':<20s} {'Mean':>10s} {'Std':>10s}  {'N':>3s}")
    for v in ALL_VARIANTS:
        s = analysis["summary"].get(v, {})
        if s.get("n", 0) > 0:
            marker = " *" if v == best else ""
            print(f"  {v:<20s} {s['mean']:>10.4f} {s['std']:>10.4f}  {s['n']:>3d}{marker}")

    # Key pairwise comparisons
    pw = analysis.get("pairwise", {})
    if "standard_vs_prmp" in pw:
        p = pw["standard_vs_prmp"]
        print(f"\n  Standard vs PRMP: Cohen's d = {p['cohens_d']:.3f}, p = {p['p_value']:.4f}")

In [ ]:
# ============================================================
# VISUALIZATION: Performance Comparison Across Tasks
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = {"standard": "#1f77b4", "prmp": "#ff7f0e", "wide": "#2ca02c",
          "auxiliary_mlp": "#d62728", "random_frozen": "#9467bd"}
short_names = {"standard": "Std", "prmp": "PRMP", "wide": "Wide",
               "auxiliary_mlp": "Aux-MLP", "random_frozen": "Rnd-Frz"}

for ax_i, (task_key, analysis) in enumerate(stat_analysis.items()):
    ax = axes[ax_i]
    means, stds, labels, clrs = [], [], [], []
    for v in ALL_VARIANTS:
        s = analysis["summary"].get(v, {})
        if s.get("n", 0) > 0:
            means.append(s["mean"])
            stds.append(s["std"])
            labels.append(short_names.get(v, v))
            clrs.append(colors.get(v, "#333333"))

    x_pos = np.arange(len(means))
    bars = ax.bar(x_pos, means, yerr=stds, capsize=4, color=clrs, alpha=0.8, edgecolor="black", linewidth=0.5)

    # Highlight best
    best_idx = int(np.argmin(means)) if analysis["lower_better"] else int(np.argmax(means))
    bars[best_idx].set_edgecolor("gold")
    bars[best_idx].set_linewidth(2.5)

    ax.set_xticks(x_pos)
    ax.set_xticklabels(labels, fontsize=9)
    metric_name = analysis["metric"].upper().replace("_", " ")
    direction = "(lower=better)" if analysis["lower_better"] else "(higher=better)"
    ax.set_ylabel(metric_name)
    ax.set_title(f"{task_key}\n{direction}", fontsize=10)
    ax.grid(True, axis="y", alpha=0.3)

plt.suptitle("PRMP Benchmark: 5 Variants x 5 Seeds per Task", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# VISUALIZATION: Prediction Scatter (Standard vs PRMP)
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax_i, ds in enumerate(data["datasets"]):
    ax = axes[ax_i]
    examples = ds["examples"]
    true_vals = [float(ex["output"]) for ex in examples]
    std_preds = [float(ex["predict_standard"]) for ex in examples]
    prmp_preds = [float(ex["predict_prmp"]) for ex in examples]

    ax.scatter(true_vals, std_preds, alpha=0.5, s=20, label="Standard", color="#1f77b4")
    ax.scatter(true_vals, prmp_preds, alpha=0.5, s=20, label="PRMP", color="#ff7f0e", marker="x")

    # Perfect prediction line
    mn, mx = min(true_vals), max(true_vals)
    ax.plot([mn, mx], [mn, mx], "k--", alpha=0.3, label="Perfect")
    ax.set_xlabel("True Value")
    ax.set_ylabel("Predicted Value")
    ax.set_title(ds["dataset"], fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("Standard vs PRMP: Predictions on Test Set", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# VISUALIZATION: Ridge R² Embedding Diagnostics & Cohen's d
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Ridge R² per FK link ---
diag = meta.get("embedding_diagnostics", [])
if diag:
    # Group by variant, collect R² values
    variant_r2 = defaultdict(list)
    link_labels = set()
    for d in diag:
        for link, vals in d.get("fk_ridge_r2", {}).items():
            variant_r2[(d["variant"], d["task"])].append((link, vals["ridge_r2"]))
            link_labels.add(link)

    # Plot for Amazon task only (first occurrence)
    amazon_diag = [d for d in diag if d["dataset"] == "rel-amazon"]
    if amazon_diag:
        variants_seen = []
        x_pos = np.arange(len(amazon_diag))
        for i, d in enumerate(amazon_diag):
            r2_vals = d.get("fk_ridge_r2", {})
            for j, (link, vals) in enumerate(r2_vals.items()):
                offset = (j - 0.5) * 0.15
                c = colors.get(d["variant"], "#333")
                ax1.bar(i + offset, vals["ridge_r2"], width=0.15, color=c, alpha=0.8,
                        label=link if i == 0 else "")
            variants_seen.append(short_names.get(d["variant"], d["variant"]))
        ax1.set_xticks(x_pos)
        ax1.set_xticklabels(variants_seen, fontsize=9)
        ax1.set_ylabel("Ridge R²")
        ax1.set_title("Embedding Diagnostics: Ridge R²\n(Amazon, review->product & review->customer)")
        ax1.grid(True, axis="y", alpha=0.3)

# --- Cohen's d heatmap for Amazon ---
amazon_analysis = stat_analysis.get("rel-amazon/review-rating", {})
pw = amazon_analysis.get("pairwise", {})
if pw:
    n_v = len(ALL_VARIANTS)
    d_matrix = np.zeros((n_v, n_v))
    for i, v1 in enumerate(ALL_VARIANTS):
        for j, v2 in enumerate(ALL_VARIANTS):
            if i == j:
                d_matrix[i, j] = 0
            else:
                key = f"{v1}_vs_{v2}"
                key2 = f"{v2}_vs_{v1}"
                if key in pw:
                    d_matrix[i, j] = pw[key]["cohens_d"]
                elif key2 in pw:
                    d_matrix[i, j] = -pw[key2]["cohens_d"]

    im = ax2.imshow(d_matrix, cmap="RdBu_r", vmin=-1.5, vmax=1.5, aspect="auto")
    ax2.set_xticks(range(n_v))
    ax2.set_xticklabels([short_names.get(v, v) for v in ALL_VARIANTS], fontsize=9, rotation=45)
    ax2.set_yticks(range(n_v))
    ax2.set_yticklabels([short_names.get(v, v) for v in ALL_VARIANTS], fontsize=9)
    ax2.set_title("Cohen's d (Amazon review-rating)\nPositive = row better than column")
    for i in range(n_v):
        for j in range(n_v):
            ax2.text(j, i, f"{d_matrix[i,j]:.2f}", ha="center", va="center", fontsize=8,
                     color="white" if abs(d_matrix[i,j]) > 0.7 else "black")
    plt.colorbar(im, ax=ax2, shrink=0.8)

plt.tight_layout()
plt.show()

# --- Per-run results table ---
print("\n" + "=" * 80)
print("PER-RUN RESULTS (all 75 runs)")
print("=" * 80)
runs = meta.get("per_run_results", [])
print(f"{'Variant':<18s} {'Task':<20s} {'Seed':>5s} {'Metric':>10s} {'Epoch':>6s} {'Params':>8s}")
for r in runs[:15]:  # Show first 15
    task = f"{r['dataset']}/{r['task']}"
    metric_val = list(r.get("test_metrics", {}).values())[0] if r.get("test_metrics") else 0
    print(f"{r['variant']:<18s} {task:<20s} {r['seed']:>5d} {metric_val:>10.4f} "
          f"{r.get('best_epoch', 0):>6d} {r.get('num_params', 0):>8d}")
print(f"... ({len(runs)} total runs)")